# Поиск объявлений с помощью двух башен

Нужно найти до 50 объявлений для каждого запроса. Сначала получим текстовые векторы с помощью E5, затем обучим две небольшие сети и построим индекс HNSW.

E5 оставляем без изменений. В модель подаём запрос с фильтрами и текст объявления. Цену, рейтинг и географию в этой версии не используем.

Ноутбук уже запускался на полном датасете. Лучший локальный Recall@50 — **0,2156** после одной эпохи. Готовый ответ лежит в `avito_retrieval/simple_artifacts/answer.csv`; для его отправки повторный запуск не нужен.

In [ ]:
# В Colab раскомментируй эти две строки. На Mac пропусти ячейку.
# from google.colab import drive
# drive.mount("/content/drive")

## Библиотеки

Установи один раз в то окружение, где открыт ноутбук.

In [ ]:
%pip install numpy pandas pyarrow torch sentence-transformers hnswlib==0.8.0

## Настройки

На Mac данные лежат в `Documents/soso/avito_retrieval/data`. В Colab сначала подключи Диск: папка `avito_retrieval` может лежать прямо в «Моём диске» или внутри `soso`.

`ENCODE_BATCH` — сколько текстов E5 обрабатывает за раз. Если не хватает памяти, уменьши до 8. `BATCH_SIZE` относится к обучению башен.

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sentence_transformers import SentenceTransformer
import hnswlib

if Path("/content").exists():
    ROOT = Path("/content/drive/MyDrive")
    if not (ROOT / "avito_retrieval").exists():
        ROOT = ROOT / "soso"
else:
    ROOT = Path.home() / "Documents/soso"
DATA = ROOT / "avito_retrieval/data"
SAVE = ROOT / "avito_retrieval/simple_artifacts"
SAVE.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
MODEL_NAME = "intfloat/multilingual-e5-small"
MAX_LENGTH = 256
ENCODE_BATCH = 16
BATCH_SIZE = 128
EPOCHS = 3
SEED = 42
print("Устройство:", DEVICE)

## Подготовка данных

У запроса берём текст и фильтры. У объявления — заголовок, параметры и описание.

В train нет query_id, поэтому различаем запросы по всем пяти поисковым признакам. Для каждого item_id оставляем первую запись. Если текст объявления менялся, более свежую версию определить не получится: времени публикации в данных нет.

In [ ]:
QUERY_COLUMNS = ["search_query", "search_location_id", "search_is_delivery_search",
         "search_infm_params_text", "search_category"]

ITEM_COLUMNS = ["item_id", "item_title_raw", "item_infm_params_text", "item_description_raw"]
train = pd.read_parquet(DATA / "train.parquet", columns=QUERY_COLUMNS + ITEM_COLUMNS)


def clean(series):
    return series.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()


def make_query_text(df):
    text = clean(df["search_query"])
    filters = clean(df["search_infm_params_text"])
    return text + (" | Фильтры: " + filters).where(filters.ne(""), "")


def make_item_text(df):
    text = clean(df["item_title_raw"])
    for col, label in [("item_infm_params_text", "Параметры"), ("item_description_raw", "Описание")]:
        value = clean(df[col])
        text = text + (" | " + label + ": " + value).where(value.ne(""), "")
    return text


train_queries = train[QUERY_COLUMNS].drop_duplicates().reset_index(drop=True)
train_queries["q_idx"] = np.arange(len(train_queries))
train_queries["text"] = make_query_text(train_queries)

train_items = train[ITEM_COLUMNS].drop_duplicates("item_id").reset_index(drop=True)
train_items["text"] = make_item_text(train_items)
item_to_idx = dict(zip(train_items["item_id"], train_items.index))

pairs = train[QUERY_COLUMNS + ["item_id"]].merge(train_queries[QUERY_COLUMNS + ["q_idx"]], on=QUERY_COLUMNS, validate="many_to_one")
pairs["i_idx"] = pairs["item_id"].map(item_to_idx)
pairs = pairs[["q_idx", "i_idx"]].drop_duplicates().astype("int64")
del train
print("Запросов:", len(train_queries), "Объявлений для валидации:", len(train_items))

## Обучение и валидация

Откладываем 10% разных текстов запросов. Один и тот же текст не должен попасть в обе части, даже если локации или фильтры отличаются.

При проверке ищем среди объявлений из train. В benchmark_items большинства правильных ответов из train нет.

In [ ]:
split_key = clean(train_queries["search_query"]).str.casefold()
rng = np.random.default_rng(SEED)
keys = rng.permutation(split_key.unique())
assert len(keys) >= 2
n_valid = max(1, int(len(keys) * 0.1))
valid_ids = set(train_queries.index[split_key.isin(keys[:n_valid])])
is_valid = pairs["q_idx"].isin(valid_ids)
train_pairs = pairs.loc[~is_valid]
valid_pairs = pairs.loc[is_valid]
print("Пар для обучения:", len(train_pairs), "для валидации:", len(valid_pairs))

## Текстовые векторы

E5 скачивается при первом запуске, дальше берём веса из папки encoder. Префиксы `query:` и `passage:` нужны самой модели.

Сохраняем векторы каждые 1000 уникальных текстов. Если выполнение прервётся, повторный запуск прочитает готовые части. Хеш выбирает папку кэша по текстам и настройкам; при ручной замене весов E5 кэш нужно удалить.

In [ ]:
encoder_path = SAVE / "encoder"
if (encoder_path / "modules.json").exists():
    encoder = SentenceTransformer(str(encoder_path), device=DEVICE, local_files_only=True)
else:
    encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)
    encoder.save(str(encoder_path))
encoder.max_seq_length = MAX_LENGTH


def encode_texts(texts, prefix):
    text_indices, unique_texts = pd.factorize(pd.Series(texts), sort=False)
    cache_key = hashlib.sha256(f"{MODEL_NAME}:{MAX_LENGTH}:{prefix}".encode())
    for text in unique_texts:
        data = text.encode("utf-8")
        cache_key.update(len(data).to_bytes(8, "little") + data)
    folder = SAVE / "cache" / cache_key.hexdigest()
    folder.mkdir(parents=True, exist_ok=True)
    parts = []
    for start in range(0, len(unique_texts), 1000):
        path = folder / f"{start}.npy"
        if not path.exists():
            vectors = encoder.encode(
                [prefix + text for text in unique_texts[start:start + 1000]],
                batch_size=ENCODE_BATCH, normalize_embeddings=True,
                convert_to_numpy=True, show_progress_bar=False,
            )
            temporary = path.with_suffix(".tmp")
            with temporary.open("wb") as file:
                np.save(file, vectors)
            temporary.replace(path)
        parts.append(np.load(path))
        print(f"{prefix} {min(start + 1000, len(unique_texts))}/{len(unique_texts)}", end="\r")
    print()
    return np.concatenate(parts)[text_indices].astype("float32")


query_embeddings = encode_texts(train_queries["text"].tolist(), "query: ")
item_embeddings = encode_texts(train_items["text"].tolist(), "passage: ")
# Освобождаем GPU от E5 на время обучения маленьких башен.
encoder.to("cpu")
if DEVICE == "mps":
    torch.mps.empty_cache()
elif DEVICE == "cuda":
    torch.cuda.empty_cache()

## Две башни

У запроса и объявления свои сети. Каждая добавляет обучаемую поправку к вектору E5, затем нормализует результат. В начале поправка равна нулю, поэтому поиск работает как исходная E5.

In [ ]:
class Tower(nn.Module):
    def __init__(self, dimension):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(dimension, 256), nn.ReLU(), nn.Linear(256, dimension)
        )
        nn.init.zeros_(self.layers[-1].weight)
        nn.init.zeros_(self.layers[-1].bias)

    def forward(self, x):
        return F.normalize(x + self.layers(x), dim=1)


def make_model():
    torch.manual_seed(SEED)
    return nn.ModuleDict({
        "query": Tower(query_embeddings.shape[1]),
        "item": Tower(item_embeddings.shape[1]),
    }).to(DEVICE)


def get_vectors(array, ids):
    return torch.from_numpy(array[ids]).to(DEVICE)


@torch.inference_mode()
def apply_tower(tower, embeddings):
    tower.eval()
    parts = []
    for start in range(0, len(embeddings), 1024):
        batch = torch.from_numpy(embeddings[start:start + 1024]).to(DEVICE)
        parts.append(tower(batch).cpu().numpy())
    return np.concatenate(parts)

## Одна эпоха обучения

Для каждого запроса выбираем случайное правильное объявление. Объявления других запросов в батче используем как отрицательные примеры. Если среди них есть ещё один известный правильный ответ, тоже помечаем его положительным.

Одинаковые текстовые входы модель не различает, поэтому их известные правильные ответы объединяем. При валидации запросы с разными фильтрами и локациями по-прежнему оцениваем отдельно.

In [ ]:
def make_positives(selected_pairs):
    joined = selected_pairs.merge(train_queries[["q_idx", "text"]], on="q_idx")
    by_text = joined.groupby("text")["i_idx"].agg(set).to_dict()
    return {int(query_id): by_text[train_queries.loc[query_id, "text"]]
            for query_id in selected_pairs["q_idx"].unique()}


def train_epoch(model, optimizer, positives, epoch):
    model.train()
    rng = np.random.default_rng(SEED + epoch)
    query_ids = rng.permutation(list(positives))
    total, count = 0.0, 0
    for start in range(0, len(query_ids), BATCH_SIZE):
        batch_query_ids = query_ids[start:start + BATCH_SIZE]
        chosen_items = [rng.choice(sorted(positives[int(query_id)])) for query_id in batch_query_ids]
        batch_item_ids = np.unique(chosen_items)
        if len(batch_item_ids) < 2:
            continue
        mask = torch.tensor([[int(i) in positives[int(query_id)] for i in batch_item_ids] for query_id in batch_query_ids],
                            dtype=torch.bool, device=DEVICE)
        query_vectors = model["query"](get_vectors(query_embeddings, batch_query_ids))
        item_vectors = model["item"](get_vectors(item_embeddings, batch_item_ids))
        scores = query_vectors @ item_vectors.T / 0.05
        log_probs = F.log_softmax(scores, dim=1)
        # У запроса может быть несколько правильных объявлений в батче.
        positive_log_probs = log_probs.masked_fill(~mask, 0).sum(1)
        positive_counts = mask.sum(1)
        loss = -(positive_log_probs / positive_counts).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
        count += 1
    if count == 0:
        raise ValueError("Нужно больше разных объявлений для обучения.")
    return total / count

## Проверка качества

Считаем долю правильных объявлений в top-50 и усредняем по запросам. Здесь поиск точный, без HNSW: приближённый индекс не должен влиять на выбор числа эпох.

Сначала проверяем исходную E5, затем сравниваем с результатом после каждой эпохи.

In [ ]:
def exact_top50(query_vectors, item_vectors):
    result = []
    k = min(50, len(item_vectors))
    for start in range(0, len(query_vectors), 32):
        scores = query_vectors[start:start + 32] @ item_vectors.T
        nearest = np.argpartition(-scores, k - 1, axis=1)[:, :k]
        # Без copy срез удерживает большой массив в памяти.
        result.extend(nearest.copy())
    return np.asarray(result)


def evaluate(model):
    truth = valid_pairs.groupby("q_idx")["i_idx"].agg(set)
    query_vectors = apply_tower(model["query"], query_embeddings[truth.index.to_numpy()])
    item_vectors = apply_tower(model["item"], item_embeddings)
    found = exact_top50(query_vectors, item_vectors)
    return float(np.mean([len(set(row) & relevant) / len(relevant)
                          for row, relevant in zip(found, truth)]))


model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
positives = make_positives(train_pairs)
best_epoch = 0
best_score = evaluate(model)
history = [{"epoch": 0, "recall50": best_score}]
print("Исходная E5:", round(best_score, 4))

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, optimizer, positives, epoch)
    score = evaluate(model)
    history.append({"epoch": epoch, "recall50": score})
    print(f"Эпоха {epoch}: loss={loss:.4f}, Recall@50={score:.4f}")
    if score > best_score:
        best_epoch, best_score = epoch, score
    pd.DataFrame(history).to_csv(SAVE / "validation.csv", index=False)

## Обучение на всех парах

Выбираем число эпох по валидации и обучаем модель заново на всём train. Если улучшения нет, оставляем нулевую поправку к E5.

После сохранения весов освобождаем память перед обработкой benchmark.

In [ ]:
model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
positives = make_positives(pairs)
for epoch in range(1, best_epoch + 1):
    loss = train_epoch(model, optimizer, positives, epoch)
    print(f"Финальное обучение {epoch}/{best_epoch}: {loss:.4f}")

torch.save({
    "state_dict": {name: value.detach().cpu() for name, value in model.state_dict().items()},
    "dimension": query_embeddings.shape[1], "epochs": best_epoch,
    "model_name": MODEL_NAME, "max_length": MAX_LENGTH,
}, SAVE / "towers.pt")
print("Сохранены башни; выбранное число эпох:", best_epoch)

# Освобождаем данные обучения перед кодированием benchmark.
import gc
del optimizer, positives, query_embeddings, item_embeddings
del train_items, pairs, train_pairs, valid_pairs, train_queries
gc.collect()
if DEVICE == "mps":
    torch.mps.empty_cache()
elif DEVICE == "cuda":
    torch.cuda.empty_cache()


## Поиск по benchmark

Считаем векторы и строим HNSW только по benchmark_items. Внутренние номера индекса затем заменим исходными item_id.

На первых 100 запросах сравниваем HNSW с точным поиском. Это проверка индекса, не Recall по скрытым правильным ответам.

In [ ]:
queries = pd.read_parquet(DATA / "benchmark_queries.parquet")
items = pd.read_parquet(DATA / "benchmark_items.parquet", columns=ITEM_COLUMNS)

assert items["item_id"].is_unique and queries["query_id"].is_unique
encoder.to(DEVICE)
benchmark_query_embeddings = encode_texts(make_query_text(queries).tolist(), "query: ")
benchmark_item_embeddings = encode_texts(make_item_text(items).tolist(), "passage: ")
encoder.to("cpu")
query_vectors = apply_tower(model["query"], benchmark_query_embeddings)
item_vectors = apply_tower(model["item"], benchmark_item_embeddings)

index = hnswlib.Index(space="cosine", dim=item_vectors.shape[1])
index.init_index(max_elements=len(item_vectors), M=32, ef_construction=200, random_seed=SEED)
index.add_items(item_vectors, np.arange(len(item_vectors)), num_threads=1)
index.set_ef(200)
labels, _ = index.knn_query(query_vectors, k=min(50, len(item_vectors)))
index.save_index(str(SAVE / "items.hnsw"))
items[["item_id"]].to_parquet(SAVE / "item_ids.parquet", index=False)

exact = exact_top50(query_vectors[:100], item_vectors)
agreement = np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(labels[:100], exact)])
print(f"Совпадение HNSW с точным top-50: {agreement:.2%}")

## Файл ответа

Одна строка на query_id, до 50 уникальных item_id через пробел. Идентификаторы сохраняем как строки, включая ведущие нули.

In [ ]:
item_ids = items["item_id"].to_numpy()
predictions = [item_ids[row].tolist() for row in labels]
assert queries["query_id"].map(lambda x: isinstance(x, str) and len(x) == 16).all()
assert items["item_id"].map(lambda x: isinstance(x, str)).all()
assert items["item_id"].str.fullmatch(r"[0-9a-f]{16}").all()
assert len(predictions) == len(queries)
assert all(len(row) <= 50 and len(row) == len(set(row)) for row in predictions)

answer = pd.DataFrame({
    "query_id": queries["query_id"],
    "answer": [" ".join(row) for row in predictions],
})
answer.to_csv(SAVE / "answer.csv", index=False, encoding="utf-8")
print("Готово:", SAVE / "answer.csv")
display(answer.head())